In [1]:
# Kaggle setup (CPU is enough): clone repo, install libraries, find the data.
# Inputs needed: competition + 02_faster_rcnn + 02b_frcnn_submit + 06_pipeline (Sofia, shared)
import os, glob, subprocess

REPO = "polluvl/hw2-video-detection"

if os.path.isdir("/kaggle/working"):
    # 1. clone the repo once per session
    if not os.path.isdir("/kaggle/working/repo"):
        r = subprocess.run(["git", "clone", "-q", f"https://github.com/{REPO}.git", "/kaggle/working/repo"],
                           capture_output=True, text=True)
        if r.returncode != 0:
            raise RuntimeError("git clone failed (is Internet on?):\n" + r.stderr)
    # 2. install the pinned libraries
    subprocess.run(["pip", "install", "-q", "-r", "/kaggle/working/repo/requirements.txt"], check=True)
    os.chdir("/kaggle/working/repo/notebooks")

    # 3. find where Kaggle mounted the competition data
    hits = glob.glob("/kaggle/input/**/test_images.csv", recursive=True)
    if not hits:
        raise RuntimeError("Competition data not found: add it in the right panel under Input")
    os.environ["DATA_ROOT"] = os.path.dirname(hits[0])
    print("data root:", os.environ["DATA_ROOT"])

data root: /kaggle/input/competitions/multiple-object-video-detection-hw-2-ucu-ai-2026


In [2]:
# 07 · Ensemble of the two best detector families with Weighted Boxes Fusion (WBF)
# Faster R-CNN (02) + YOLO pipeline with tiles + temporal rescoring (06).
# No retraining: we fuse predictions that were already saved by 02, 02b and 06.
# Weights and the WBF IoU threshold are chosen on the step-5 val frames (same frames as 02-05).
import glob
import sys
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

sys.path.insert(0, str(Path("../src").resolve()))
from common import (evaluate, find_data_root, list_frames, list_test_frames, load_preds,
                    save_preds, work_dir, write_submission)
from inference import merge_wbf

# WBF warns once per zero-size box; silence it to keep the notebook output short
warnings.filterwarnings("ignore", message="Zero area box skipped")

DATA = find_data_root()
OUT = work_dir()


def find(name):
    hits = [h for h in glob.glob(f"/kaggle/input/**/{name}", recursive=True) if "/competitions/" not in h]
    if not hits:
        raise RuntimeError(f"{name} not found: add the notebook that produced it as input")
    return hits[0]


# saved predictions: dict image_key -> (N, 6) [x1, y1, x2, y2, score, class]
frcnn_val = load_preds(find("preds_val_frcnn_base.npz"))           # 02, step-5 val frames
yolo_val_all = load_preds(find("preds_valall_pipe_temporal.npz"))  # 06, all val frames
frcnn_test = load_preds(find("preds_test_frcnn_base.npz"))         # 02b
yolo_test = load_preds(find("preds_test_final.npz"))               # 06 (tiles + temporal)

val5 = list_frames(DATA / "val", step=5)
yolo_val = {k: yolo_val_all[k] for k in val5.image_key if k in yolo_val_all}
print("val frames:", len(val5), "| test frames:", len(frcnn_test), len(yolo_test))

EMPTY = np.zeros((0, 6), np.float32)
TOPK = 300  # keep the 300 most confident boxes per model and image (keeps WBF fast)


def top(arr):
    return arr[np.argsort(-arr[:, 4])[:TOPK]] if len(arr) else arr


def ensemble(frames, p_frcnn, p_yolo, weights, iou_thr):
    out = {}
    for r in frames.itertuples():
        a = top(p_frcnn.get(r.image_key, EMPTY))
        b = top(p_yolo.get(r.image_key, EMPTY))
        out[r.image_key] = merge_wbf([a, b], r.width, r.height, iou_thr=iou_thr, weights=list(weights))
    return out


# 1. single models on the same val frames (reference)
rows = [{"name": "frcnn_base", **evaluate(val5, frcnn_val)},
        {"name": "yolo_pipe_temporal", **evaluate(val5, yolo_val)}]

# 2. small grid over model weights and WBF IoU threshold
for weights in [(1, 1), (2, 1), (3, 1)]:
    for iou_thr in [0.5, 0.6]:
        t0 = time.time()
        fused = ensemble(val5, frcnn_val, yolo_val, weights, iou_thr)
        rows.append({"name": f"wbf_w{weights[0]}{weights[1]}_iou{iou_thr}", "weights": weights,
                     "iou_thr": iou_thr, **evaluate(val5, fused)})
        print(rows[-1]["name"], rows[-1]["mAP"], f"({time.time() - t0:.0f}s)")

table = pd.DataFrame(rows)
table.to_csv(OUT / "ensemble_results.csv", index=False)
table[["name", "mAP", "AP50", "AP75", "AP_small", "AP_medium", "AP_large"]]

val frames: 571 | test frames: 3811 3811
wbf_w11_iou0.5 0.2654 (81s)
wbf_w11_iou0.6 0.2713 (91s)
wbf_w21_iou0.5 0.2595 (78s)
wbf_w21_iou0.6 0.2652 (91s)
wbf_w31_iou0.5 0.2567 (78s)
wbf_w31_iou0.6 0.2625 (92s)


,name,mAP,AP50,AP75,AP_small,AP_medium,AP_large
0,frcnn_base,0.2384,0.4763,0.2070,0.1033,0.2209,0.4263
1,yolo_pipe_temporal,0.2356,0.4500,0.2143,0.1201,0.2054,0.4089
2,wbf_w11_iou0.5,0.2654,0.5118,0.2389,0.1272,0.2382,0.4590
3,wbf_w11_iou0.6,0.2713,0.5187,0.2479,0.1342,0.2450,0.4663
4,wbf_w21_iou0.5,0.2595,0.5059,0.2319,0.1215,0.2364,0.4500
5,wbf_w21_iou0.6,0.2652,0.5130,0.2403,0.1267,0.2427,0.4560
6,wbf_w31_iou0.5,0.2567,0.5023,0.2304,0.1183,0.2344,0.4473
7,wbf_w31_iou0.6,0.2625,0.5101,0.2379,0.1231,0.2412,0.4530


In [3]:
# 3. write a submission only if the best ensemble beats Faster R-CNN alone on val
best = table[table.name.str.startswith("wbf")].sort_values("mAP").iloc[-1]
frcnn_map = table.set_index("name").loc["frcnn_base", "mAP"]
print(f"best ensemble: {best['name']} val mAP {best.mAP:.4f} vs Faster R-CNN alone {frcnn_map:.4f}")

if best.mAP > frcnn_map:
    test_frames = list_test_frames(DATA)
    fused_test = ensemble(test_frames, frcnn_test, yolo_test, best.weights, best.iou_thr)
    save_preds(fused_test, OUT / "preds_test_ensemble.npz")
    sub = write_submission(test_frames, fused_test, OUT / "submission.csv")
    # format check against the official list of test images
    assert len(sub) == len(test_frames) and set(sub.image_id) == set(test_frames.image_id)
    print(f"submission.csv written: {len(sub)} rows")
else:
    print("ensemble is not better on val -> no new submission (keep Faster R-CNN)")

best ensemble: wbf_w11_iou0.6 val mAP 0.2713 vs Faster R-CNN alone 0.2384
submission.csv written: 3811 rows
